# Federated Learning with Adaptive Epochs and Weight Rejection
This notebook implements federated learning with adaptive epochs for bottom performers and weight rejection.

**Strategy:**
1. All 100 clients train for 10 fixed epochs each round
2. Bottom 10 clients get adaptive training (up to 50 total epochs) if they didn't improve
3. All clients: Accept weights only if test accuracy improved, else reject
4. FedAvg and proceed to next round

**Key Features:**
- Efficient training with minimal overhead
- Smart weight rejection based on test accuracy
- Adaptive epochs only for struggling clients
- Common test dataset of 500 samples

In [17]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import os
from tqdm import tqdm

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.10.0


In [18]:
# GPU Configuration
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU detected and configured")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠ No GPU detected - Running on CPU")
print("=" * 60 + "\n")

GPU CONFIGURATION
TensorFlow version: 2.10.0
Num GPUs Available: 1
✓ GPU detected and configured



## Configuration

In [19]:
# Federated Learning Configuration
NUM_CLIENTS = 100
NUM_ROUNDS = 100          # SAME as original weight rejection notebook
FIXED_EPOCHS = 10         # Fixed epochs for all clients each round
BATCH_SIZE = 32           # SAME as original (32 not 64)

# Adaptive Training Parameters
NUM_BOTTOM_CLIENTS = 10   # Number of lowest performers to get adaptive training
MAX_ADAPTIVE_EPOCHS = 50  # Maximum total epochs (including fixed 10)
ADAPTIVE_EPOCHS_STEP = 10 # Train in steps of 10 epochs during adaptive phase

# Directories
DATA_DIR = 'mnist_100_clients_2'
RESULTS_DIR = 'results_adaptive_weight_rejection'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/plots', exist_ok=True)

print("=" * 70)
print("FEDERATED LEARNING: ADAPTIVE EPOCHS + WEIGHT REJECTION")
print("=" * 70)
print(f"Number of Clients: {NUM_CLIENTS}")
print(f"Communication Rounds: {NUM_ROUNDS}")
print(f"Fixed Epochs per Round: {FIXED_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print()
print("Adaptive Training:")
print(f"  Bottom {NUM_BOTTOM_CLIENTS} clients get adaptive training")
print(f"  Max total epochs: {MAX_ADAPTIVE_EPOCHS}")
print(f"  Adaptive step: {ADAPTIVE_EPOCHS_STEP} epochs")
print()
print("Weight Rejection:")
print(f"  Accept only if test accuracy improves")
print(f"  Else reject and keep previous best weights")
print()
print(f"Data Directory: {DATA_DIR}/")
print(f"Results Directory: {RESULTS_DIR}/")
print("=" * 70 + "\n")

FEDERATED LEARNING: ADAPTIVE EPOCHS + WEIGHT REJECTION
Number of Clients: 100
Communication Rounds: 100
Fixed Epochs per Round: 10
Batch Size: 32

Adaptive Training:
  Bottom 10 clients get adaptive training
  Max total epochs: 50
  Adaptive step: 10 epochs

Weight Rejection:
  Accept only if test accuracy improves
  Else reject and keep previous best weights

Data Directory: mnist_100_clients_2/
Results Directory: results_adaptive_weight_rejection/



## Load Data

In [20]:
# Load test data (common for all clients)
print("Loading common test dataset...")
test_file = os.path.join(DATA_DIR, 'test_500_samples.npz')
test_data = np.load(test_file)

x_test = test_data['x'] / 255.0
y_test = test_data['y']
x_test = x_test.reshape(len(x_test), 28*28)

print(f"✓ Test data loaded: {x_test.shape}")
print(f"  Labels shape: {y_test.shape}")

Loading common test dataset...
✓ Test data loaded: (500, 784)
  Labels shape: (500,)


In [21]:
# Load all client data
print(f"\nLoading data for {NUM_CLIENTS} clients...")
client_data = []

for client_id in range(1, NUM_CLIENTS + 1):
    client_file = os.path.join(DATA_DIR, f'client_{client_id}.npz')
    data = np.load(client_file)
    
    x_client = data['x'] / 255.0
    y_client = data['y']
    x_client = x_client.reshape(len(x_client), 28*28)
    
    client_data.append({
        'x_train': x_client,
        'y_train': y_client,
        'x_test': x_test,
        'y_test': y_test
    })
    
    if client_id % 20 == 0:
        print(f"  Loaded {client_id}/{NUM_CLIENTS} clients")

print(f"\n✓ All {NUM_CLIENTS} clients loaded successfully")
print(f"  Each client: {len(client_data[0]['x_train'])} training samples")
print(f"  Common test set: {len(x_test)} samples\n")


Loading data for 100 clients...
  Loaded 20/100 clients
  Loaded 40/100 clients
  Loaded 60/100 clients
  Loaded 80/100 clients
  Loaded 100/100 clients

✓ All 100 clients loaded successfully
  Each client: 100 training samples
  Common test set: 500 samples



## Model Architecture

In [22]:
# Define model - SAME as original weight rejection (simpler, proven to work)
def create_model():
    """Lightweight model optimized for small datasets"""
    model = keras.Sequential([
        keras.layers.Dense(64, input_shape=(784,), activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax")
    ])
    
    # Compile with SAME hyperparameters as original
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),  # Higher LR than before
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Test model
print("Testing model architecture...")
test_model = create_model()
test_model.summary()
print("\n✓ Model architecture validated (same as working version)\n")

Testing model architecture...
Model: "sequential_10939"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_32817 (Dense)         (None, 64)                50240     
                                                                 
 dropout_21878 (Dropout)     (None, 64)                0         
                                                                 
 dense_32818 (Dense)         (None, 32)                2080      
                                                                 
 dropout_21879 (Dropout)     (None, 32)                0         
                                                                 
 dense_32819 (Dense)         (None, 10)                330       
                                                                 
Total params: 52,650
Trainable params: 52,650
Non-trainable params: 0
_________________________________________________________________

✓ Model archite

## Helper Functions

In [23]:
# Federated Averaging function
def federated_averaging(weights_list):
    """Average weights from all clients (FedAvg)"""
    avg_weights = []
    for weights_tuple in zip(*weights_list):
        avg_weights.append(np.mean(weights_tuple, axis=0))
    return avg_weights

print("✓ Federated averaging function defined")

✓ Federated averaging function defined


## Initialize Training

In [24]:
# Initialize global model and tracking
print("=" * 60)
print("INITIALIZING FEDERATED LEARNING")
print("=" * 60)

global_model = create_model()
global_weights = global_model.get_weights()

# Tracking arrays
client_test_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_best_weights = [None for _ in range(NUM_CLIENTS)]
client_best_test_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_rejections = [[] for _ in range(NUM_CLIENTS)]
client_epochs_used = [[] for _ in range(NUM_CLIENTS)]
client_adaptive_rounds = [[] for _ in range(NUM_CLIENTS)]

print("✓ Global model initialized")
print("✓ Tracking arrays created")
print("✓ Weight rejection + adaptive epochs ready\n")

INITIALIZING FEDERATED LEARNING
✓ Global model initialized
✓ Tracking arrays created
✓ Weight rejection + adaptive epochs ready



## Main Training Loop

In [ ]:
# Main federated training loop
print("=" * 70)
print("STARTING FEDERATED TRAINING")
print("=" * 70 + "\n")

for round_num in tqdm(range(NUM_ROUNDS), desc="Communication Rounds", unit="round"):
    print(f"\n{'='*70}")
    print(f"ROUND {round_num + 1}/{NUM_ROUNDS}")
    print(f"{'='*70}")
    
    local_weights = []
    round_test_accs = []
    round_rejections = 0
    round_acceptances = 0
    round_client_results = []  # Store (client_id, test_acc, epochs_used) for sorting
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 1: Fixed Training for ALL Clients
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 1: Training all {NUM_CLIENTS} clients for {FIXED_EPOCHS} epochs...")
    
    client_round_data = {}  # Store results before weight rejection decision
    
    for client_id in tqdm(range(NUM_CLIENTS), desc="Fixed Training", unit="client", leave=False):
        # Create fresh model
        client_model = create_model()
        client_model.set_weights(global_weights)
        
        # Get client's data
        x_train = client_data[client_id]['x_train']
        y_train = client_data[client_id]['y_train']
        x_test = client_data[client_id]['x_test']
        y_test = client_data[client_id]['y_test']
        
        # Store old accuracy for comparison
        old_test_acc = client_best_test_acc[client_id]
        
        # Train for fixed epochs (SIMPLE, FAST)
        client_model.fit(
            x_train, y_train,
            epochs=FIXED_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        
        # Evaluate on test data
        _, new_test_acc = client_model.evaluate(x_test, y_test, verbose=0)
        new_weights = [w.copy() for w in client_model.get_weights()]
        
        # Store results temporarily
        client_round_data[client_id] = {
            'old_acc': old_test_acc,
            'new_acc': new_test_acc,
            'new_weights': new_weights,
            'epochs_used': FIXED_EPOCHS,
            'got_adaptive': False
        }
        
        # Store for sorting (to find bottom performers)
        round_client_results.append((client_id, new_test_acc, FIXED_EPOCHS))
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 2: Adaptive Training for Bottom Performers
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 2: Adaptive training for bottom {NUM_BOTTOM_CLIENTS} clients...")
    
    # Sort by test accuracy (ascending - lowest first)
    round_client_results.sort(key=lambda x: x[1])
    bottom_clients = [cid for cid, _, _ in round_client_results[:NUM_BOTTOM_CLIENTS]]
    
    print(f"Bottom {NUM_BOTTOM_CLIENTS} clients identified:")
    for i, (cid, acc, _) in enumerate(round_client_results[:NUM_BOTTOM_CLIENTS]):
        print(f"  {i+1}. Client {cid+1}: {acc*100:.2f}%")
    
    adaptive_improved = 0
    for client_id in tqdm(bottom_clients, desc="Adaptive Training", unit="client", leave=False):
        data = client_round_data[client_id]
        target_acc = data['old_acc']
        current_acc = data['new_acc']
        
        # Only do adaptive training if they haven't improved yet
        if current_acc <= target_acc and target_acc > 0:
            # Create model with current weights
            client_model = create_model()
            client_model.set_weights(data['new_weights'])
            
            x_train = client_data[client_id]['x_train']
            y_train = client_data[client_id]['y_train']
            x_test = client_data[client_id]['x_test']
            y_test = client_data[client_id]['y_test']
            
            best_adaptive_acc = current_acc
            best_adaptive_weights = data['new_weights']
            total_epochs = FIXED_EPOCHS
            
            # Train in steps until improvement or max epochs
            while total_epochs < MAX_ADAPTIVE_EPOCHS:
                client_model.fit(
                    x_train, y_train,
                    epochs=ADAPTIVE_EPOCHS_STEP,
                    batch_size=BATCH_SIZE,
                    verbose=0
                )
                
                total_epochs += ADAPTIVE_EPOCHS_STEP
                _, test_acc = client_model.evaluate(x_test, y_test, verbose=0)
                
                # Track best
                if test_acc > best_adaptive_acc:
                    best_adaptive_acc = test_acc
                    best_adaptive_weights = [w.copy() for w in client_model.get_weights()]
                
                # Stop if improved beyond target
                if test_acc > target_acc:
                    adaptive_improved += 1
                    break
            
            # Update client data with best adaptive results
            client_round_data[client_id]['new_acc'] = best_adaptive_acc
            client_round_data[client_id]['new_weights'] = best_adaptive_weights
            client_round_data[client_id]['epochs_used'] = total_epochs
            client_round_data[client_id]['got_adaptive'] = True
    
    if adaptive_improved > 0:
        print(f"  ✓ {adaptive_improved}/{len(bottom_clients)} clients improved with adaptive training")
    
    # ═══════════════════════════════════════════════════════════
    # PHASE 3: Weight Rejection Decision
    # ═══════════════════════════════════════════════════════════
    print(f"\nPhase 3: Weight rejection decision...")
    
    for client_id in range(NUM_CLIENTS):
        data = client_round_data[client_id]
        old_acc = data['old_acc']
        new_acc = data['new_acc']
        epochs_used = data['epochs_used']
        
        # DECISION: Accept only if improved
        if new_acc > old_acc:
            # ACCEPT
            client_best_weights[client_id] = data['new_weights']
            client_best_test_acc[client_id] = new_acc
            client_rejections[client_id].append(0)
            round_acceptances += 1
            final_acc = new_acc
        else:
            # REJECT - keep old weights
            if client_best_weights[client_id] is None:
                # First round - accept anyway
                client_best_weights[client_id] = data['new_weights']
                client_best_test_acc[client_id] = new_acc
                client_rejections[client_id].append(0)
                round_acceptances += 1
                final_acc = new_acc
            else:
                # Reject, use previous best
                client_rejections[client_id].append(1)
                round_rejections += 1
                final_acc = client_best_test_acc[client_id]
        
        # Track results
        client_test_acc_history[client_id].append(final_acc)
        client_epochs_used[client_id].append(epochs_used)
        client_adaptive_rounds[client_id].append(1 if data['got_adaptive'] else 0)
        round_test_accs.append(final_acc)
        
        # Collect best weights for FedAvg
        local_weights.append(client_best_weights[client_id])
    
    # ═══════════════════════════════════════════════════════════
    # Federated Averaging
    # ═══════════════════════════════════════════════════════════
    global_weights = federated_averaging(local_weights)
    global_model.set_weights(global_weights)
    
    # Round summary
    avg_test_acc = np.mean(round_test_accs) * 100
    min_test_acc = np.min(round_test_accs) * 100
    max_test_acc = np.max(round_test_accs) * 100
    std_test_acc = np.std([acc * 100 for acc in round_test_accs])
    total_adaptive = sum([1 for cid in range(NUM_CLIENTS) if client_round_data[cid]['got_adaptive']])
    
    print(f"\n📊 Round {round_num + 1} Summary:")
    print(f"   Avg Test Accuracy: {avg_test_acc:.2f}%")
    print(f"   Std Dev: {std_test_acc:.2f}%")
    print(f"   Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")
    print(f"   Weights Accepted: {round_acceptances}/{NUM_CLIENTS}")
    print(f"   Weights Rejected: {round_rejections}/{NUM_CLIENTS}")
    print(f"   Adaptive Training: {total_adaptive}/{NUM_CLIENTS} clients")

print("\n" + "="*70)
print("FEDERATED TRAINING COMPLETE!")
print("="*70 + "\n")

STARTING FEDERATED TRAINING



Communication Rounds:   0%|          | 0/100 [00:00<?, ?round/s]


ROUND 1/100

Phase 1: Training all 100 clients for 10 epochs...



Phase 2: Adaptive training for bottom 10 clients...
Bottom 10 clients identified:
  1. Client 53: 32.40%
  2. Client 57: 34.40%
  3. Client 71: 35.00%
  4. Client 34: 36.00%
  5. Client 62: 41.40%
  6. Client 90: 42.40%
  7. Client 32: 42.60%
  8. Client 48: 42.60%
  9. Client 81: 42.60%
  10. Client 93: 43.20%


Communication Rounds:   1%|          | 1/100 [00:53<1:28:33, 53.68s/round]

...


Phase 3: Weight rejection decision...

📊 Round 64 Summary:
   Avg Test Accuracy: 90.65%
   Std Dev: 0.66%
   Range: [88.20%, 92.20%]
   Weights Accepted: 5/100
   Weights Rejected: 95/100
   Adaptive Training: 10/100 clients

ROUND 65/100

Phase 1: Training all 100 clients for 10 epochs...



Phase 2: Adaptive training for bottom 10 clients...
Bottom 10 clients identified:
  1. Client 40: 86.60%
  2. Client 64: 87.00%
  3. Client 17: 88.00%
  4. Client 21: 88.00%
  5. Client 79: 88.00%
  6. Client 61: 88.20%
  7. Client 76: 88.20%
  8. Client 53: 88.40%
  9. Client 87: 88.40%
  10. Client 97: 88.40%


Communication Rounds:  65%|██████▌   | 65/100 [1:09:55<41:36, 71.32s/round]


Phase 3: Weight rejection decision...

📊 Round 65 Summary:
   Avg Test Accuracy: 90.67%
   Std Dev: 0.67%
   Range: [88.20%, 92.20%]
   Weights Accepted: 4/100
   Weights Rejected: 96/100
   Adaptive Training: 10/100 clients

ROUND 66/100

Phase 1: Training all 100 clients for 10 epochs...


Communication Rounds:  65%|██████▌   | 65/100 [1:10:30<37:57, 65.08s/round]


## Results Analysis

In [ ]:
# Calculate final statistics
final_test_accs = [client_test_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]
total_rejections_per_client = [sum(client_rejections[i]) for i in range(NUM_CLIENTS)]
total_epochs_per_client = [sum(client_epochs_used[i]) for i in range(NUM_CLIENTS)]
total_adaptive_rounds = [sum(client_adaptive_rounds[i]) for i in range(NUM_CLIENTS)]

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"Average Final Test Accuracy: {np.mean(final_test_accs):.2f}%")
print(f"Std Dev: {np.std(final_test_accs):.2f}%")
print(f"Test Accuracy Range: [{np.min(final_test_accs):.2f}%, {np.max(final_test_accs):.2f}%]")
print()
print("Weight Rejection Statistics:")
print(f"  Total Rejections: {sum(total_rejections_per_client)} / {NUM_CLIENTS * NUM_ROUNDS}")
print(f"  Rejection Rate: {sum(total_rejections_per_client) / (NUM_CLIENTS * NUM_ROUNDS) * 100:.2f}%")
print(f"  Avg Rejections per Client: {np.mean(total_rejections_per_client):.2f}")
print()
print("Adaptive Training Statistics:")
print(f"  Total Adaptive Rounds: {sum(total_adaptive_rounds)}")
print(f"  Avg Adaptive Rounds per Client: {np.mean(total_adaptive_rounds):.2f}")
print(f"  Total Epochs Used: {sum(total_epochs_per_client)}")
print(f"  Avg Epochs per Client: {np.mean(total_epochs_per_client):.1f}")
print("=" * 70 + "\n")

FINAL RESULTS
Average Final Test Accuracy: 91.16%
Std Dev: 0.69%
Test Accuracy Range: [89.60%, 92.60%]

Weight Rejection Statistics:
  Total Rejections: 8374 / 10000
  Rejection Rate: 83.74%
  Avg Rejections per Client: 83.74

Adaptive Training Statistics:
  Total Adaptive Rounds: 937
  Avg Adaptive Rounds per Client: 9.37
  Total Epochs Used: 136890
  Avg Epochs per Client: 1368.9



In [ ]:
# Detailed per-client results
print("PER-CLIENT RESULTS:")
print("-" * 110)
print(f"{'Client':<8} {'Final Acc':<12} {'Total Epochs':<14} {'Rejections':<14} {'Adaptive Rounds':<16}")
print("-" * 110)

for client_id in range(NUM_CLIENTS):
    final_acc = final_test_accs[client_id]
    total_epochs = total_epochs_per_client[client_id]
    rejections = total_rejections_per_client[client_id]
    adaptive = total_adaptive_rounds[client_id]
    print(f"{client_id + 1:<8} {final_acc:.2f}%{'':<6} {total_epochs:<14} {rejections:<14} {adaptive:<16}")

print("-" * 110 + "\n")

PER-CLIENT RESULTS:
--------------------------------------------------------------------------------------------------------------
Client   Final Acc    Total Epochs   Rejections     Adaptive Rounds 
--------------------------------------------------------------------------------------------------------------
1        90.40%       1520           84             13              
2        91.60%       1000           83             0               
3        91.40%       1050           86             2               
4        92.40%       1040           85             1               
5        91.00%       1400           85             10              
6        92.00%       1000           82             0               
7        90.80%       1680           84             17              
8        91.40%       1040           85             1               
9        91.60%       1090           83             3               
10       92.20%       1040           84             1               

## Visualization

In [ ]:
# Plot 1: Average accuracy across all clients
print("Creating average accuracy plot...")

plt.figure(figsize=(12, 7))
rounds = range(1, NUM_ROUNDS + 1)

avg_accs = []
std_accs = []
for r in range(NUM_ROUNDS):
    round_accs = [client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]
    avg_accs.append(np.mean(round_accs))
    std_accs.append(np.std(round_accs))

plt.plot(rounds, avg_accs, marker='o', linewidth=3, markersize=8, 
         color='#2E86AB', label='Average Test Accuracy')
plt.fill_between(rounds, 
                 np.array(avg_accs) - np.array(std_accs),
                 np.array(avg_accs) + np.array(std_accs),
                 alpha=0.3, color='#A8DADC', label='±1 Std Dev')

plt.title('Average Test Accuracy - Adaptive Epochs + Weight Rejection', 
          fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Test Accuracy (%)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([50, 100])

plt.savefig(f'{RESULTS_DIR}/plots/average_accuracy.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: average_accuracy.png")
plt.close()

Creating average accuracy plot...
✓ Saved: average_accuracy.png


In [ ]:
# Plot 2: Individual client accuracy curves (subplots)
print("Creating individual client accuracy plots (10x10 grid)...")

fig, axes = plt.subplots(10, 10, figsize=(30, 30))
fig.suptitle('Test Accuracy per Client - Adaptive Epochs + Weight Rejection', 
             fontsize=20, fontweight='bold', y=0.995)

rounds = range(1, NUM_ROUNDS + 1)

for client_id in range(NUM_CLIENTS):
    row = client_id // 10
    col = client_id % 10
    ax = axes[row, col]
    
    accs = [acc * 100 for acc in client_test_acc_history[client_id]]
    final_acc = accs[-1]
    improvement = final_acc - accs[0]
    rejections = total_rejections_per_client[client_id]
    adaptive = total_adaptive_rounds[client_id]
    
    color = '#2E86AB' if improvement >= 0 else '#E63946'
    ax.plot(rounds, accs, marker='o', linewidth=2, markersize=4, color=color)
    ax.fill_between(rounds, accs, alpha=0.3, color=color)
    
    title_color = 'green' if improvement >= 0 else 'red'
    ax.set_title(f'C{client_id+1} ({improvement:+.1f}%) R:{rejections} A:{adaptive}', 
                fontsize=9, fontweight='bold', color=title_color)
    
    ax.text(NUM_ROUNDS, final_acc, f'{final_acc:.0f}%', 
            fontsize=8, fontweight='bold', verticalalignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    ax.set_xlim(0.5, NUM_ROUNDS + 0.5)
    
    if row == 9:
        ax.set_xlabel('Round', fontsize=8)
    if col == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=8)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plots/all_clients_grid.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: all_clients_grid.png")
plt.close()

Creating individual client accuracy plots (10x10 grid)...
✓ Saved: all_clients_grid.png


In [ ]:
# Plot 3: Rejection heatmap
print("Creating weight rejection heatmap...")

plt.figure(figsize=(14, 10))

rejection_matrix = np.zeros((NUM_CLIENTS, NUM_ROUNDS))
for i in range(NUM_CLIENTS):
    for j in range(NUM_ROUNDS):
        rejection_matrix[i, j] = client_rejections[i][j]

plt.imshow(rejection_matrix, cmap='RdYlGn_r', aspect='auto', interpolation='nearest')
plt.colorbar(label='Rejected (1=Yes, 0=No)')
plt.title('Weight Rejection Status per Client per Round', fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Client ID', fontsize=13)
plt.tight_layout()

plt.savefig(f'{RESULTS_DIR}/plots/rejection_heatmap.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: rejection_heatmap.png")
plt.close()

Creating weight rejection heatmap...
✓ Saved: rejection_heatmap.png


In [ ]:
# Plot 4: Epochs distribution
print("Creating epochs distribution plot...")

plt.figure(figsize=(12, 7))

epochs_matrix = []
for r in range(NUM_ROUNDS):
    round_epochs = [client_epochs_used[i][r] for i in range(NUM_CLIENTS)]
    epochs_matrix.append(round_epochs)

bp = plt.boxplot(epochs_matrix, positions=range(1, NUM_ROUNDS + 1), 
                 widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor='lightblue', color='blue'),
                 medianprops=dict(color='red', linewidth=2))

plt.title('Epochs Distribution per Round (Fixed + Adaptive)', 
          fontsize=16, fontweight='bold')
plt.xlabel('Communication Round', fontsize=13)
plt.ylabel('Epochs Used', fontsize=13)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

plt.savefig(f'{RESULTS_DIR}/plots/epochs_distribution.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: epochs_distribution.png")
plt.close()

print("\n✅ All plots generated!\n")

Creating epochs distribution plot...
✓ Saved: epochs_distribution.png

✅ All plots generated!



## Save Results

In [ ]:
# Save results and model
print("Saving results...")

results_data = {
    'client_test_acc_history': client_test_acc_history,
    'client_rejections': client_rejections,
    'client_epochs_used': client_epochs_used,
    'client_adaptive_rounds': client_adaptive_rounds,
    'final_test_accs': final_test_accs,
    'config': {
        'num_clients': NUM_CLIENTS,
        'num_rounds': NUM_ROUNDS,
        'fixed_epochs': FIXED_EPOCHS,
        'max_adaptive_epochs': MAX_ADAPTIVE_EPOCHS,
        'num_bottom_clients': NUM_BOTTOM_CLIENTS,
        'batch_size': BATCH_SIZE
    }
}

np.save(f'{RESULTS_DIR}/training_results.npy', results_data, allow_pickle=True)
print(f"✓ Results saved: {RESULTS_DIR}/training_results.npy")

# Save global model
global_model.save(f'{RESULTS_DIR}/global_model.h5')
print(f"✓ Model saved: {RESULTS_DIR}/global_model.h5")

print("\n✅ COMPLETE! All results saved to:", RESULTS_DIR)

Saving results...
✓ Results saved: results_adaptive_weight_rejection/training_results.npy
✓ Model saved: results_adaptive_weight_rejection/global_model.h5

✅ COMPLETE! All results saved to: results_adaptive_weight_rejection
